# Lab 9: OpenAlex LSA and Clustering

**COMPSS 211 | Fall 2026 | Student copy**

Create lightweight document embeddings and inspect clusters after the embeddings lecture.

**Date/deadline:** Friday, November 6, 2026

Work through each task and replace the response placeholders with your own answers.

## Scenario

A teammate has reduced a small set of text features with LSA and clustered the resulting document vectors. Reproduce the steps, read the source abstracts, and decide whether any cluster label is defensible.

## Goal

- Apply TruncatedSVD (LSA).
- Cluster with fixed randomness.
- Inspect abstracts before naming clusters.

## Keep handy

- **Required:** LSA and KMeans.
- **Optional:** A guarded Word2Vec extension may be attempted in a separate environment.

In [ ]:
from pathlib import Path
import json
import os
import sys

import pandas as pd
import numpy as np
from IPython.display import display

REQUIRED_PYTHON = "3.12.13"
if sys.version.split()[0] != REQUIRED_PYTHON:
    raise RuntimeError(
        f"This notebook requires Python {REQUIRED_PYTHON}; "
        f"the active kernel is {sys.version.split()[0]}."
    )

COURSE_MARKERS = ("data", "homework", "lab")

def is_course_root(candidate):
    return all((candidate / name).is_dir() for name in COURSE_MARKERS)

def locate_course_root():
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if is_course_root(candidate):
            return candidate

    in_colab = (
        "google.colab" in sys.modules
        or "COLAB_RELEASE_TAG" in os.environ
    )
    if in_colab:
        matches = [
            candidate
            for candidate in Path("/content").iterdir()
            if candidate.is_dir() and is_course_root(candidate)
        ]
        if len(matches) == 1:
            return matches[0]

    raise FileNotFoundError(
        "Course repository not found. Run from a cloned copy of "
        "macss-berkeley/compss-211a. In Colab, clone or upload the "
        "complete repository under /content, then rerun this cell."
    )

COURSE_ROOT = locate_course_root()
DATA_DIR = COURSE_ROOT / "data"
GENERATED_DIR = COURSE_ROOT / "generated"
GENERATED_DIR.mkdir(parents=True, exist_ok=True)
print(f"Python {sys.version.split()[0]} | data={DATA_DIR}")

## Practice: LSA embedding and cautious cluster labels

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
works = pd.read_csv(DATA_DIR / "openalex_berkeley_abstracts_2024_sample.csv").dropna(subset=["abstract"])
vectorizer = TfidfVectorizer(stop_words="english", min_df=2, max_features=1500)
matrix = vectorizer.fit_transform(works["abstract"])
lsa = TruncatedSVD(n_components=8, random_state=211)
embeddings = lsa.fit_transform(matrix)
clusterer = KMeans(n_clusters=5, random_state=211, n_init=30)
works["cluster"] = clusterer.fit_predict(embeddings)
display(works.groupby("cluster").agg(
    documents=("openalex_id", "count"),
    example_domain=("primary_domain", lambda values: values.mode().iat[0]),
))

### Optional extension

Word2Vec may be explored in a separate environment. It is not part
of the pinned course dependencies and is not required for completion.

### Your notes

Before you leave, write down one thing you can now do and one question you still have.

> Write your notes here.

## Exit

The last 10 minutes are reserved for the three-question quiz on Monday's material. Use the lab to practice the ideas before you answer from memory.